In [3]:
from transformers import AutoTokenizer,AutoModelForMultipleChoice,TrainingArguments,Trainer
from datasets import  load_dataset

In [4]:
dataset = load_dataset("c3","dialog")
dataset

Using the latest cached version of the dataset since c3 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'dialog' at /Users/ryan.pang/.cache/huggingface/datasets/c3/dialog/0.0.0/28e91a21a22b95987a90a46cb6d7741c7aad8158 (last modified on Sun Jun 22 10:35:14 2025).


DatasetDict({
    train: Dataset({
        features: ['documents', 'document_id', 'questions'],
        num_rows: 4885
    })
    test: Dataset({
        features: ['documents', 'document_id', 'questions'],
        num_rows: 1627
    })
    validation: Dataset({
        features: ['documents', 'document_id', 'questions'],
        num_rows: 1628
    })
})

In [5]:
dataset["train"]["documents"][0]

['男：你今天晚上有时间吗?我们一起去看电影吧?', '女：你喜欢恐怖片和爱情片，但是我喜欢喜剧片，科幻片一般。所以……']

In [6]:
dataset["train"]["questions"][0]

{'question': ['女的最喜欢哪种电影?'],
 'answer': ['喜剧片'],
 'choice': [['恐怖片', '爱情片', '喜剧片', '科幻片']]}

In [7]:
tokenizer = AutoTokenizer.from_pretrained("hfl/chinese-macbert-base",user_fast = True)
tokenizer

BertTokenizerFast(name_or_path='hfl/chinese-macbert-base', vocab_size=21128, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [26]:
#需要把每一条处理成 context - question - choice - label 
# 有N多个答案候选 就得处理成N条

def process_func(examples):
    context = []
    question_choices = []
    labels = []

    for question, ctx in zip(examples["questions"], examples["documents"]):
        q_text = question["question"][0]
        choices = question["choice"][0]
        answer = question["answer"][0]

        for choice in choices:
            context.append('\n'.join(ctx))
            question_choices.append(q_text + " " + choice)

        if len(choices) < 4:
            for _ in range(4 - len(choices)):
                context.append('\n'.join(ctx))
                question_choices.append(q_text + " " + "无")

        labels.append(choices.index(answer))
    
    tokenized_examples = tokenizer(context,question_choices,truncation='only_first',max_length=128,padding='max_length')
    
    tokenized_examples = {k: [v[i: i + 4] for i in range(0, len(v), 4)] for k, v in tokenized_examples.items()}
    
    tokenized_examples['labels'] = labels
    # 
    return tokenized_examples

In [27]:
# tokenized_c3 = dataset.map(process_func,batched=True)
# tokenized_c3
process_func(dataset['train'][:1])

key: input_ids
value: [[[101, 4511, 8038, 872, 791, 1921, 3241, 677, 3300, 3198, 7313, 1408, 136, 2769, 812, 671, 6629, 1343, 4692, 4510, 2512, 1416, 136, 1957, 8038, 872, 1599, 3614, 2607, 2587, 4275, 1469, 4263, 2658, 4275, 8024, 852, 3221, 2769, 1599, 3614, 1599, 1196, 4275, 8024, 4906, 2404, 4275, 671, 5663, 511, 2792, 809, 100, 100, 102, 1957, 4638, 3297, 1599, 3614, 1525, 4905, 4510, 2512, 136, 2607, 2587, 4275, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 4511, 8038, 872, 791, 1921, 3241, 677, 3300, 3198, 7313, 1408, 136, 2769, 812, 671, 6629, 1343, 4692, 4510, 2512, 1416, 136, 1957, 8038, 872, 1599, 3614, 2607, 2587, 4275, 1469, 4263, 2658, 4275, 8024, 852, 3221, 2769, 1599, 3614, 1599, 1196, 4275, 8024, 4906, 2404, 4275, 671, 5663, 511, 2792, 809, 100, 100, 102, 1957, 4638, 3297, 1599, 3614, 1525, 4905, 4510, 2512, 136, 4263, 2658, 4275, 

In [29]:
print(tokenized_c3['train'][0])

{'documents': ['男：你今天晚上有时间吗?我们一起去看电影吧?', '女：你喜欢恐怖片和爱情片，但是我喜欢喜剧片，科幻片一般。所以……'], 'document_id': '25-35', 'questions': {'question': ['女的最喜欢哪种电影?'], 'answer': ['喜剧片'], 'choice': [['恐怖片', '爱情片', '喜剧片', '科幻片']]}, 'input_ids': [[101, 4511, 8038, 872, 791, 1921, 3241, 677, 3300, 3198, 7313, 1408, 136, 2769, 812, 671, 6629, 1343, 4692, 4510, 2512, 1416, 136, 1957, 8038, 872, 1599, 3614, 2607, 2587, 4275, 1469, 4263, 2658, 4275, 8024, 852, 3221, 2769, 1599, 3614, 1599, 1196, 4275, 8024, 4906, 2404, 4275, 671, 5663, 511, 2792, 809, 100, 100, 102, 1957, 4638, 3297, 1599, 3614, 1525, 4905, 4510, 2512, 136, 2607, 2587, 4275, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [101, 4511, 8038, 872, 791, 1921, 3241, 677, 3300, 3198, 7313, 1408, 136, 2769, 812, 671, 6629, 1343, 4692, 4510, 2512, 1416, 136, 1957, 8038, 872, 1599, 3614, 2607, 2587, 4275, 1469, 4263, 2658, 4275, 

In [31]:
print(tokenizer.encode(['男：你今天晚上有时间吗?我们一起去看电影吧?', '女：你喜欢恐怖片和爱情片，但是我喜欢喜剧片，科幻片一般。所以……']))

[101, 4511, 8038, 872, 791, 1921, 3241, 677, 3300, 3198, 7313, 1408, 136, 2769, 812, 671, 6629, 1343, 4692, 4510, 2512, 1416, 136, 102, 1957, 8038, 872, 1599, 3614, 2607, 2587, 4275, 1469, 4263, 2658, 4275, 8024, 852, 3221, 2769, 1599, 3614, 1599, 1196, 4275, 8024, 4906, 2404, 4275, 671, 5663, 511, 2792, 809, 100, 100, 102]


In [20]:
model = AutoModelForMultipleChoice.from_pretrained("hfl/chinese-macbert-base")

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at hfl/chinese-macbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
import numpy as np
import evaluate
accuracy = evaluate.load("accuracy")

def compute_metric(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [22]:
args = TrainingArguments(
    output_dir="./muliple_choice",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True
)

In [23]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_c3["train"].select(range(20)),
    eval_dataset=tokenized_c3["validation"].select(range(20)),
    compute_metrics=compute_metric
)

/var/folders/zv/x4vdjf_9115_6x3p0ybt2qzjzgmldp/T/ipykernel_2607/2691934113.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/opt/anaconda3/envs/llm/lib/python3.10/site-packages/accelerate/accelerator.py:449: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/opt/anaconda3/envs/llm/lib/python3.10/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


In [24]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.384912,0.250000


TrainOutput(global_step=2, training_loss=1.44915771484375, metrics={'train_runtime': 154.7655, 'train_samples_per_second': 0.129, 'train_steps_per_second': 0.013, 'total_flos': 10524347719680.0, 'train_loss': 1.44915771484375, 'epoch': 1.0})

In [50]:
from typing import Any
import torch


class MultipleChoicePipeline:
    def __init__(self,model,tokenizer):
        self.tokenizer = tokenizer
        self.model = model
        self.model.to('cpu')
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    def preprocess(self,context,question,choices):
        cs , qcs = [],[]
        for choice in choices:
            cs.append(context)
            qcs.append(question+" "+choice)
        return self.tokenizer(cs,qcs,truncation='only_first',max_length=256,return_tensors="pt")
    
    def predict(self,inputs):
        inputs = {k: v.unsqueeze(0).to(self.device) for k, v in inputs.items()}
        return self.model(**inputs).logits
    
    def postprocess(self,logits,choices):
        predition = torch.argmax(logits, dim=-1).cpu().item()
        return choices[predition]
    
    def __call__(self, context, question, choices):
        inputs = self.preprocess(context,question,choices)
        logits = self.predict(inputs)
        result = self.postprocess(logits,choices)
        return result




In [51]:
pipe = MultipleChoicePipeline(model,tokenizer)

In [52]:
pipe("小明在北京上班", "小明在哪里上班？", ["北京", "上海", "河北", "海南", "河北", "海南"])

'上海'